In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.max_columns", None)


In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Shape of dataset:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


In [ ]:
print("Target variable value counts:")
df["Churn"].value_counts()


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:", df.duplicated().sum())


In [ ]:
# TotalCharges is object type — force conversion to numeric, invalid values become NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing values in TotalCharges after conversion:", df["TotalCharges"].isnull().sum())
df[df["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]


In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print("Missing values remaining:", df["TotalCharges"].isnull().sum())


In [ ]:
# customerID is a unique identifier with no predictive value — drop it
df = df.drop(columns=["customerID"])

# SeniorCitizen is stored as 0/1 but every other Yes/No column is stored as text.
# Converting it to Yes/No keeps encoding consistent across all categorical columns.
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

print("Final shape after cleaning:", df.shape)
df.head()


In [ ]:
plt.figure()
ax = sns.countplot(data=df, x="Churn", hue="Churn", palette="Set2", legend=False)
ax.set_title("Churn Distribution")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.savefig("plot_churn_distribution.png", dpi=120)
plt.show()

print(df["Churn"].value_counts(normalize=True) * 100)


In [ ]:
plt.figure(figsize=(7,4))
ax = sns.countplot(data=df, x="Contract", hue="Churn", palette="Set2")
ax.set_title("Churn by Contract Type")
plt.tight_layout()
plt.savefig("plot_churn_by_contract.png", dpi=120)
plt.show()


In [ ]:
plt.figure()
sns.histplot(data=df, x="tenure", hue="Churn", multiple="stack", bins=30, palette="Set2")
plt.title("Tenure Distribution by Churn")
plt.tight_layout()
plt.savefig("plot_tenure_churn.png", dpi=120)
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numeric Features")
plt.tight_layout()
plt.savefig("plot_correlation_heatmap.png", dpi=120)
plt.show()


In [ ]:
# Encode target variable: Yes -> 1, No -> 0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Separate features (X) and target (y)
X = df.drop(columns=["Churn"])
y = df["Churn"]

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)


In [ ]:
# One-hot encode all categorical columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print("Shape before encoding:", X.shape)
print("Shape after encoding:", X_encoded.shape)
X_encoded.head()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)
print("Logistic Regression trained.")


In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
print("Random Forest trained.")


In [ ]:
y_pred_log = log_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test)

def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred) 
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"--- {name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print()
    return acc, prec, rec, f1

log_scores = evaluate("Logistic Regression", y_test, y_pred_log)
rf_scores = evaluate("Random Forest", y_test, y_pred_rf)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

cm_log = confusion_matrix(y_test, y_pred_log)
sns.heatmap(cm_log, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
axes[0].set_title("Confusion Matrix — Logistic Regression")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens", ax=axes[1],
            xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
axes[1].set_title("Confusion Matrix — Random Forest")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.savefig("plot_confusion_matrices.png", dpi=120)
plt.show()


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [log_scores[0], rf_scores[0]],
    "Precision": [log_scores[1], rf_scores[1]],
    "Recall": [log_scores[2], rf_scores[2]],
    "F1-Score": [log_scores[3], rf_scores[3]],
})
comparison


In [ ]:
# Show predictions for a few real test-set customers, compared to their actual label
sample_idx = X_test.index[:5]
results = pd.DataFrame({
    "Actual": y_test.loc[sample_idx].map({0: "No Churn", 1: "Churn"}),
    "Predicted (Random Forest)": pd.Series(rf_model.predict(X_test.loc[sample_idx]), index=sample_idx).map({0: "No Churn", 1: "Churn"})
})
results


In [ ]:
# Manually create ONE new sample customer and predict Churn / No Churn
new_customer = {
    "gender": "Female", "SeniorCitizen": "No", "Partner": "Yes", "Dependents": "No",
    "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
    "StreamingMovies": "Yes", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 95.0, "TotalCharges": 190.0,
}

new_df = pd.DataFrame([new_customer])
new_encoded = pd.get_dummies(new_df)
# align columns with training data (missing dummy columns filled with 0)
new_encoded = new_encoded.reindex(columns=X_encoded.columns, fill_value=0)

prediction = rf_model.predict(new_encoded)[0]
probability = rf_model.predict_proba(new_encoded)[0][1]

print("New customer profile:", new_customer)
print(f"\nPrediction: {'Churn' if prediction == 1 else 'No Churn'}")
print(f"Churn probability: {probability:.2%}")
